# DORA Percentage Deployment Failure Rates

## Pre-Requisites

In [ ]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    df_deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(df_deployments.info())
    print("\nFirst 5 rows:")
    print(df_deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")

In [ ]:
# Add new row and quater columns
df_deployments["month"] = df_deployments["deployed_at"].dt.month

df_deployments.head(100)

## Tinkering

In [ ]:
df_fail_rates = df_deployments[['application_id','month','status']]
df_fail_rates

In [ ]:
df_count = df_deployments.groupby(['month','application_id','status']).agg({'status':'count'})

print(df_count)

In [ ]:
df_app2 = df_count.xs('app002', level='application_id')

print(df_app2)

In [ ]:
df_app2['Percentage'] = 100 * df_app2['status'] / df_app2.groupby(level=0)['status'].transform('sum')



print(df_app2_indexed)

In [ ]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	data_frame=df_app2,
	title="Total Deployments by Status",
	x="month",
	y="Percentage",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig.update_layout(barmode='stack')

fig_month_stat_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()

## Just app002

In [ ]:
df_app2_only = df_deployments[df_deployments['application_id'] == 'app002']

# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html
df_app2_only.reset_index(drop=True, inplace=True)

df_app2_only

In [ ]:
df_simple_app2_only = df_deployments[df_deployments['application_id'] == 'app002'][['month','status']]
df_simple_app2_only.reset_index(drop=True, inplace=True)
df_simple_app2_only

In [ ]:
df_app2_simple = df_app2_only[['month','status']]

df_app2_simple

In [ ]:
df_app2_grouped = df_app2_simple.groupby(['month','status']).agg({'status':'count'})

print(df_app2_grouped)

In [ ]:
df_app2_percentage = df_app2_grouped.groupby(level=0).apply(
	lambda x: 100 * x / x.sum())

# Fix the index (drop the duplicate month level)
df_app2_percentage.index = df_app2_percentage.index.droplevel(1)

# Rename the column to avoid conflict during reset_index
df_app2_percentage = df_app2_percentage.rename(columns={'status': 'percentage'})

print("df_app2_percentage:")
print(df_app2_percentage)
print("Columns:", df_app2_percentage.columns)
print("Index names:", df_app2_percentage.index.names)

# Reset index to create a DataFrame suitable for plotting
df_app2_perentage_df = df_app2_percentage.reset_index()

print(df_app2_perentage_df)

In [ ]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	data_frame=df_app2_perentage_df,
	title="Deployment Failure Rates by Month",
	x="month",
	y="Percentage",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_bar.update_layout(barmode='stack')

fig_month_stat_bar.update_yaxes(
	title_text="Percentage (%)"
)

fig_month_stat_bar.update_xaxes(
	title_text="Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()